# Uniform random subsample of another run's final set (`subsample_of`, `subsample_n`)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
STAGE = "ancestry_round2"
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
parent = load_run(cfg.param("subsample_of"))
src = single_target(parent, "ancestry_round2")
N, SEED = cfg.param("subsample_n"), cfg.param("seed")
ids = [l.strip() for l in get_store(src).read_text(src).splitlines() if l.strip()]
keep = sorted(np.random.default_rng(SEED).choice(ids, size=min(N, len(ids)), replace=False))
rep.note(subsample_of=parent.run_id, subsample_n=N, seed=SEED)
rep.count("n_parent", len(ids)); print("sampled", rep.count("n_kept", len(keep)), "of", len(ids))
rep.check("sample size as asked", len(keep) == N, f"{len(keep)}")
if needs_run(OUT, FORCE):
    with record(cfg, STAGE, OUT) as meta:
        get_store(OUT).write_text(OUT, "\n".join(keep) + "\n")
        meta.update(n_kept=len(keep), subsample_of=parent.run_id, seed=SEED)
rep.save()